# Tutorial 8: Using Prefab Pipelines

Building a pipeline from scratch gives you full control, but PhenoTypic
also ships **prefab pipelines** — pre-configured `ImagePipeline` subclasses
tuned for common organisms and plate types. In this tutorial you will
survey the available prefabs, apply one, and compare results.

**What you will learn:**

1. What prefab pipelines are and when to use them
2. Survey the available prefabs
3. Apply a prefab pipeline
4. Compare results from different prefabs

## Imports

In [ ]:
from phenotypic.data import load_yeast_plate

## Available Prefab Pipelines

Each prefab is an `ImagePipeline` subclass with operations and measurements
already configured. Choose based on your organism and plate conditions.

| Prefab | Best for | Strategy |
|--------|----------|----------|
| `HeavyOtsuPipeline` | General-purpose yeast, clean plates | Multi-stage Otsu with refinement |
| `HeavyWatershedPipeline` | Touching or clustered colonies | Watershed segmentation |
| `RoundPeaksPipeline` | Round colonies, lightweight | Peak detection |
| `HeavyRoundPeaksPipeline` | Round colonies needing refinement | Extended peak detection |
| `FilamentousFungiPipeline` | Filamentous fungi (*Neurospora*, etc.) | BM3D denoise + specialized detector |
| `GridSectionPipeline` | Pre-tiled grid sections | Section-level processing |

## Apply HeavyOtsuPipeline

Let's start with the most general-purpose option — `HeavyOtsuPipeline`.
It chains Gaussian blur, CLAHE, median filtering, Sobel edge enhancement,
Otsu detection, and several refinement steps (morphological opening,
border removal, small-object removal, mask fill).

In [ ]:
from phenotypic.prefab import HeavyOtsuPipeline

plate = load_yeast_plate()
heavy_otsu = HeavyOtsuPipeline()
result_otsu = heavy_otsu.apply(plate)
result_otsu.dash(overlay=True)

In [ ]:
print(f"HeavyOtsuPipeline detected {result_otsu.num_objects} colonies")

## Apply RoundPeaksPipeline

Now let's try `RoundPeaksPipeline` — a lighter approach that uses peak
detection to find circular colonies. It is faster but may miss irregular
or faint colonies.

In [ ]:
from phenotypic.prefab import RoundPeaksPipeline

plate2 = load_yeast_plate()
round_peaks = RoundPeaksPipeline()
result_rp = round_peaks.apply(plate2)
result_rp.dash(overlay=True)

In [ ]:
print(f"RoundPeaksPipeline detected {result_rp.num_objects} colonies")

## Compare

Different prefabs produce different results on the same plate. The best
choice depends on your colonies, your imaging conditions, and what you
need to measure. A quick visual comparison and colony count helps you
decide.

In [ ]:
print(f"HeavyOtsuPipeline:   {result_otsu.num_objects} colonies")
print(f"RoundPeaksPipeline:  {result_rp.num_objects} colonies")

## Prefabs Include Measurements

Prefab pipelines come with measurements pre-configured, so you can call
`.apply_and_measure()` directly — no need to add your own `meas` list.

In [ ]:
plate3 = load_yeast_plate()
df = heavy_otsu.apply_and_measure(plate3)
print(f"{len(df)} colonies measured across {df.shape[1]} features")
df.head()

## Summary

Prefab pipelines are the fastest path from plate image to results:

- **`HeavyOtsuPipeline`** — robust general-purpose detection with refinement
- **`RoundPeaksPipeline`** — lightweight peak-based detection for round colonies
- **`HeavyWatershedPipeline`** — for touching or clustered colonies
- **`FilamentousFungiPipeline`** — specialized for branching fungal morphology
- All prefabs support `.apply()`, `.apply_and_measure()`, `.to_json()`, etc.

Choose based on your organism, plate conditions, and desired accuracy.
When a prefab is close but not quite right, use it as a starting point
and customize the parameters.

**Next up:** [Tutorial 9: Diagnosing Image Quality](09_diagnosing_image_quality.ipynb) —
assess plate quality before choosing a pipeline.